In [ ]:
import sys
from pathlib import Path

sys.dont_write_bytecode = True
CASE_NAME = "ex006_PHT3D_06"
candidates = (
    candidate
    for base in (Path.cwd(), *Path.cwd().parents)
    for candidate in (base, base / "examples" / CASE_NAME)
)
CASE_DIR = next(
    (
        path.resolve()
        for path in candidates
        if path.name == CASE_NAME and (path / "modflow_model.py").is_file()
    ),
    None,
)
if CASE_DIR is None:
    raise FileNotFoundError(f"Cannot locate examples/{CASE_NAME} from {Path.cwd()}")
EXAMPLES_DIR = CASE_DIR.parent
if str(EXAMPLES_DIR) not in sys.path:
    sys.path.insert(0, str(EXAMPLES_DIR))

from example_utils import load_results, restore_archives, runtime_path

CASE_FILE = CASE_DIR / "run.py"
INPUT_DIR = CASE_DIR / "input_data"
OUTPUT_DIR = runtime_path(CASE_FILE, "output")
SIMULATION_DIR = runtime_path(CASE_FILE, "simulation")
restore_archives(OUTPUT_DIR)
restore_archives(SIMULATION_DIR)


In [2]:
import matplotlib.pyplot as plt
import numpy as np

data, headings, result_times = load_results(OUTPUT_DIR)
data.shape


(851, 3, 180)

In [ ]:
reference = np.load(INPUT_DIR / "PHT3D_06_results.npy", allow_pickle=False)
observations = np.load(INPUT_DIR / "observations.npy", allow_pickle=False)
save_times = reference["time_days"]
times_meas = observations["time_minutes"]
Ca_meas, T_meas, Na_meas = (observations[name] for name in ("Ca", "T", "Na"))


myresults = data

myca = myresults[:, headings.index("Ca"), -1] * 1000
myt = myresults[:, headings.index("T"), -1] * 1000
myna = myresults[:, headings.index("Na"), -1] * 1000


fig = plt.figure(figsize=(8, 5.5))
ax = fig.add_subplot(3, 1, 1)
ax.plot(save_times * 24 * 60, np.interp(save_times, result_times, myca), "m", label="Ca")
ax.plot(save_times * 24 * 60, reference["Ca"] * 1000, "m--", label="PHT3D")
ax.plot(times_meas, Ca_meas, "mo", label="Ca (observed)")
ax.grid(True)
ax.set_xlim([0, 300])
ax.set_ylim([0, 2])
ax.tick_params(axis="x", labelsize=12)
ax.tick_params(axis="y", labelsize=12)
ax.set_ylabel("Ca (mmol/L)", fontsize=12)
ax.legend(fontsize=12)

ax = fig.add_subplot(3, 1, 2)
ax.plot(save_times * 24 * 60, np.interp(save_times, result_times, myt), "r", label="Tenside")
ax.plot(save_times * 24 * 60, reference["T"] * 1000, "r--", label="PHT3D")
ax.plot(times_meas, T_meas, "ro", label="Tenside (observed)")
ax.grid(True)
ax.set_xlim([0, 300])
ax.set_ylim([0, 5])
ax.tick_params(axis="x", labelsize=12)
ax.tick_params(axis="y", labelsize=12)
ax.set_ylabel("T (mmol/L)", fontsize=12)
ax.legend(fontsize=12)

ax = fig.add_subplot(3, 1, 3)
ax.plot(save_times * 24 * 60, np.interp(save_times, result_times, myna), "b", label="Na")
ax.plot(save_times * 24 * 60, reference["Na"] * 1000, "b--", label="PHT3D")
ax.plot(times_meas, Na_meas, "bo", label="Na (observed)")
ax.grid(True)
ax.set_xlim([0, 300])
ax.set_ylim([0, 10])
ax.set_xlabel("time (min)", fontsize=12)
ax.tick_params(axis="x", labelsize=12)
ax.tick_params(axis="y", labelsize=12)
ax.set_ylabel("Na (mmol/L)", fontsize=12)
ax.legend(fontsize=12)
plt.show()
